# Phase 1: Load & Explore

Load the IEEE-CIS Fraud Detection data, merge transaction + identity tables, and profile the dataset.

In [ ]:
import os
import pandas as pd
import numpy as np

DATA_DIR = "/Users/shengqu/Desktop/CSCI5253/FinalProject/ieee-fraud-detection" 
OUT_DIR =  "/Users/shengqu/Desktop/CSCI5253/FinalProject/ieee-fraud-detection/data" 

## 1. Load Data

In [2]:
print("Loading train_transaction...")
train_txn = pd.read_csv(os.path.join(DATA_DIR, "train_transaction.csv"))
print(f"  shape: {train_txn.shape}")

print("Loading train_identity...")
train_id = pd.read_csv(os.path.join(DATA_DIR, "train_identity.csv"))
print(f"  shape: {train_id.shape}")

Loading train_transaction...
  shape: (590540, 394)
Loading train_identity...
  shape: (144233, 41)


In [ ]:
# Left-join with identity rows
df = train_txn.merge(train_id, on="TransactionID", how="left")
print(f"Merged shape: {df.shape}")
print(f"Rows with identity info: {df['id_01'].notna().sum():,} "
      f"({df['id_01'].notna().mean():.1%})")
df.head(3)

Merged shape: (590540, 434)
Rows with identity info: 144,233 (24.4%)


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Class Imbalance

In [4]:
fraud_rate = df["isFraud"].mean()
print(f"Fraud rate: {fraud_rate:.4%} "
      f"({df['isFraud'].sum():,} fraud / {len(df):,} total)")

df["isFraud"].value_counts().rename({0: "Legit", 1: "Fraud"})

Fraud rate: 3.4990% (20,663 fraud / 590,540 total)


isFraud
Legit    569877
Fraud     20663
Name: count, dtype: int64

## 3. Check missing values

In [5]:
entity_cols = [
    "card1", "card2", "card3", "card4", "card5", "card6",
    "addr1", "addr2",
    "P_emaildomain", "R_emaildomain",
    "DeviceInfo", "id_30", "id_31", "id_33",
]

rows = []
for c in entity_cols:
    if c in df.columns:
        rows.append({
            "column": c,
            "missing_%": round(df[c].isna().mean() * 100, 2),
            "unique_values": df[c].nunique()
        })

pd.DataFrame(rows).set_index("column")

,missing_%,unique_values
column,,
card1,0.00,13553
card2,1.51,500
card3,0.27,114
card4,0.27,4
card5,0.72,119
card6,0.27,4
addr1,11.13,332
addr2,11.13,74
P_emaildomain,15.99,59


## 4. Time Structure

In [6]:
span_days = (df["TransactionDT"].max() - df["TransactionDT"].min()) / 86400
print(f"TransactionDT range: {df['TransactionDT'].min()} to {df['TransactionDT'].max()}")
print(f"Approx span: {span_days:.0f} days")

TransactionDT range: 86400 to 15811131
Approx span: 182 days


## 5. Fraud Rate by ProductCD

In [7]:
df.groupby("ProductCD")["isFraud"].agg(["mean", "count"]).rename(
    columns={"mean": "fraud_rate", "count": "n_transactions"}
).sort_values("fraud_rate", ascending=False).round(4)

,fraud_rate,n_transactions
ProductCD,,
C,0.1169,68519
S,0.0590,11628
H,0.0477,33024
R,0.0378,37699
W,0.0204,439670


## 6. Save Merged Parquet

In [ ]:
out_path = os.path.join(OUT_DIR, "train_merged.parquet")
df.to_parquet(out_path, index=False)
print(f"Saved to {out_path}")

Saved to /Users/shengqu/Desktop/CSCI5253/FinalProject/ieee-fraud-detection/data/train_merged.parquet
Next step: 02_baseline_xgboost.ipynb
